# LAB-HW-04 — Clock, Reset, and Physical I/O

**One problem today: run stateful RTL from a real platform clock/reset and map its output to the board through constraints.**

Prerequisite: LAB-HW-03 passed. Still no Linux and no AXI today.

**Project Trace:** RMD-012A · T-HW-004/T-HW-011

## 1. Three connections added today

<svg xmlns="http://www.w3.org/2000/svg" width="860" height="310" viewBox="0 0 860 310" role="img" aria-label="LAB-HW-04 clock reset and physical output map">
  <rect x="30" y="35" width="175" height="95" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="118" y="68" text-anchor="middle" font-size="15">Zynq UltraScale+ PS</text>
  <text x="118" y="93" text-anchor="middle" font-size="12">pl_clk0</text>
  <text x="118" y="113" text-anchor="middle" font-size="12">pl_resetn0</text>
  <rect x="300" y="160" width="175" height="80" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="388" y="192" text-anchor="middle" font-size="15">proc_sys_reset</text>
  <text x="388" y="216" text-anchor="middle" font-size="12">peripheral_aresetn</text>
  <rect x="550" y="35" width="170" height="95" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="635" y="68" text-anchor="middle" font-size="15">kv260_blink_core</text>
  <text x="635" y="93" text-anchor="middle" font-size="12">counter / resetn</text>
  <text x="635" y="113" text-anchor="middle" font-size="12">bank45_gpio[4:0]</text>
  <rect x="550" y="205" width="170" height="65" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="635" y="233" text-anchor="middle" font-size="14">XDC / package pins</text>
  <text x="635" y="254" text-anchor="middle" font-size="12">J11 J10 K13 F11 A12</text>
  <rect x="760" y="205" width="80" height="65" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="800" y="233" text-anchor="middle" font-size="14">Bank45</text>
  <text x="800" y="254" text-anchor="middle" font-size="12">visible I/O</text>
  <path d="M205 78 L550 78" stroke="#333" stroke-width="2"/><polygon points="550,78 540,73 540,83" fill="#333"/>
  <text x="375" y="63" text-anchor="middle" font-size="12">pl_clk0</text>
  <path d="M205 112 L300 190" stroke="#333" stroke-width="2"/><polygon points="300,190 290,184 292,196" fill="#333"/>
  <text x="248" y="157" text-anchor="middle" font-size="12">pl_resetn0</text>
  <path d="M475 200 L585 130" stroke="#333" stroke-width="2"/><polygon points="585,130 573,131 579,140" fill="#333"/>
  <text x="535" y="180" text-anchor="middle" font-size="12">design-local resetn</text>
  <path d="M635 130 L635 205" stroke="#333" stroke-width="2"/><polygon points="635,205 630,195 640,195" fill="#333"/>
  <path d="M720 238 L760 238" stroke="#333" stroke-width="2"/><polygon points="760,238 750,233 750,243" fill="#333"/>
</svg>

For the first time, the design explicitly contains:

- **clock source**: PS `pl_clk0`;
- **reset source**: PS `pl_resetn0` → `proc_sys_reset`;
- **physical mapping**: `bank45_gpio[*]` → XDC → K26 package pin.

The PS is only a clock/reset provider today, not a runtime software host.

## 2. Stateful logic: why the marker becomes a blink

`kv260_blink_core` contains a 26-bit counter.

With a nominal 100 MHz clock, the highest counter bit changes slowly enough to observe. Output contract:

- `bank45_gpio[0]` = highest counter bit, periodic;
- `bank45_gpio[4:1]` remains a marker.

This is Lesson 4's “registers need a clock” on real FPGA hardware.

## 3. Keep reset layers separate

<svg xmlns="http://www.w3.org/2000/svg" width="820" height="170" viewBox="0 0 820 170" role="img" aria-label="LAB-HW-04 reset layers">
  <rect x="20" y="50" width="150" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="95" y="80" text-anchor="middle" font-size="15">PS pl_resetn0</text>
  <text x="95" y="102" text-anchor="middle" font-size="12">platform source</text>
  <rect x="235" y="50" width="170" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="320" y="80" text-anchor="middle" font-size="15">proc_sys_reset</text>
  <text x="320" y="102" text-anchor="middle" font-size="12">reset synchronizer</text>
  <rect x="470" y="50" width="165" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="552" y="80" text-anchor="middle" font-size="14">peripheral_aresetn</text>
  <text x="552" y="102" text-anchor="middle" font-size="12">design-local reset</text>
  <rect x="700" y="50" width="100" height="70" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="750" y="80" text-anchor="middle" font-size="14">blink core</text>
  <text x="750" y="102" text-anchor="middle" font-size="12">resetn</text>
  <path d="M170 85 L235 85 M405 85 L470 85 M635 85 L700 85" stroke="#333" stroke-width="2"/>
  <polygon points="235,85 225,80 225,90" fill="#333"/>
  <polygon points="470,85 460,80 460,90" fill="#333"/>
  <polygon points="700,85 690,80 690,90" fill="#333"/>
</svg>

`resetn` is active-low.

SW2 is a SOM-level hard reset. Pressing SW2 can trigger an upstream system reset sequence, but:

**SW2 is not a wire directly connected to `kv260_blink_core.resetn`.**

If you cannot explain that distinction, the Human Check has not passed.

## 4. Now inspect the XDC

Open:

`boards/kv260/constraints/bank45_gpio.xdc`

For example:

```tcl
set_property PACKAGE_PIN J11 [get_ports {bank45_gpio[0]}]
set_property IOSTANDARD LVCMOS33 [get_ports {bank45_gpio[0]}]
```

The first line answers:

> Which K26 package pin receives this logical port?

The second answers:

> Which electrical I/O standard applies?

In SystemVerilog, `bank45_gpio[0]` is only a bit. The constraint makes it physical I/O.

### Checkpoint A — Physical mapping before clock/reset

Before running Vivado, stop and verify only the **logical-port → physical-pin** layer:

1. Point to the XDC line that maps `bank45_gpio[0]` to package pin J11.
2. Explain why `IOSTANDARD LVCMOS33` is an electrical constraint rather than SystemVerilog behavior.
3. State which facts come from the course RTL and which come from the KV260 board mapping.

If these three answers are unclear, stay here. Do not add clock/reset debugging yet.

## 5. Build: PS clock/reset + PL core

From the repository root:

```bash
vivado -mode batch -nojournal \
  -log lab-hw-04-build.log \
  -source boards/kv260/scripts/build_lab04_blink.tcl
```

The script:

1. creates a KV260/K26 Vivado project;
2. creates Zynq UltraScale+ PS using the KV260 board preset;
3. creates `proc_sys_reset`;
4. wires `pl_clk0` / `pl_resetn0` into the reset controller and `kv260_blink_core`;
5. generates the wrapper;
6. runs synthesis → implementation → bitstream;
7. emits timing/resource reports;
8. rejects a build with no clock, missing setup/hold timing paths, negative setup slack, or negative hold slack.

Expected artifact:

`build/kv260/lab-hw-04/kv260_blink.bit`

## 6. Program + Observe

Compute SHA-256 first, then reuse LAB-HW-03's programming helper:

```bash
vivado -mode batch -nojournal \
  -log lab-hw-04-program.log \
  -source boards/kv260/scripts/program_bitstream.tcl \
  -tclargs build/kv260/lab-hw-04/kv260_blink.bit
```

**Expected Evidence:**

- `xck26*` programming success;
- visible periodic activity on Bank45 bit 0;
- the other marker bits remain stable;
- timing/resource report exists;
- bitstream hash is recorded.

The periodic change matters: it demonstrates clock-driven state evolution rather than a static power-on state.

## 7. What timing evidence means here

The build log records the PS→PL clock/reset source and the timing report records implementation timing analysis.

You do not need full clocking architecture yet. Know only:

- counter registers change on real clock edges;
- a clock is not an ordinary data wire;
- timing analysis needs the clock domain;
- this Lab treats timing as a pass/fail oracle: both worst setup slack and worst hold slack must be non-negative;
- if the design is held in reset, the visible output will not blink.

The build helper prints `TIMING_SETUP_WORST_SLACK_NS` and `TIMING_HOLD_WORST_SLACK_NS` and returns non-zero on a negative value. Do not skip the timing report merely because programming succeeded.

### Checkpoint B — Clock/reset/timing before programming

After the build completes, verify the **state-evolution** layer separately:

- a real Vivado clock exists;
- setup and hold timing paths exist;
- `TIMING_SETUP_WORST_SLACK_NS` is non-negative;
- `TIMING_HOLD_WORST_SLACK_NS` is non-negative;
- the reset path in the log is `pl_resetn0 -> proc_sys_reset/peripheral_aresetn -> blink_core/resetn`.

Only after Checkpoint A **and** Checkpoint B pass should you program the board.

## 8. Save Evidence

T-HW-004 evidence includes at least:

- `lab-hw-04-build.log`;
- `timing_summary.rpt`;
- `utilization.rpt`;
- `kv260_blink.bit` SHA-256;
- `lab-hw-04-program.log`;
- blink observation/photo/video;
- explanation of at least one logical-bit → package-pin XDC mapping;
- carrier revision, Vivado version, Git commit.

Continue using `boards/kv260/evidence/manifest.example.json` for the local evidence manifest.

## 9. If it does not work

Debug the newly added layers:

1. **LAB-HW-03 marker cannot program** → stop; the prerequisite failed;
2. **board part / PS IP cannot be created** → return to LAB-HW-00 board definitions;
3. **synthesis/implementation fails** → inspect the build log, not JTAG;
4. **program succeeds but output never changes** → inspect clock/reset wiring, especially `pl_clk0` and `peripheral_aresetn`;
5. **only some physical outputs look wrong** → inspect XDC / carrier revision / physical polarity;
6. **SW2 changes behavior** → that is system-reset evidence; do not rewrite it as “I directly asserted RTL resetn.”

## 10. Human Check

1. Which abstraction layers contain `PACKAGE_PIN J11` and SystemVerilog `bank45_gpio[0]`?
2. Why is `IOSTANDARD LVCMOS33` not a code-style setting?
3. Why are `pl_resetn0` and `peripheral_aresetn` different abstraction layers?
4. Why must SW2 not be taught as “the RTL reset button”?
5. If programming succeeds but the counter does not move, do you inspect clock/reset or the neuron algorithm first? Why?
6. What is the one major new concept added by LAB-HW-04 over LAB-HW-03?

## 11. Official basis

- AMD UG1089 — Vivado Board Flow  
  https://docs.amd.com/r/en-US/ug1089-kv260-starter-kit/Vivado-Board-Flow
- AMD UG1089 — Board Reset  
  https://docs.amd.com/r/en-US/ug1089-kv260-starter-kit/Board-Reset
- AMD/Xilinx Board Store — KV260 SOM board part 1.4 / K26 package mapping  
  https://github.com/Xilinx/XilinxBoardStore/tree/master/boards/Xilinx/kv260_som/1.4
- AMD/Xilinx KV260 examples use K26 `xck26-sfvc784-2LV-c`, the KV260 board part, and PS `pl_clk0` for PL clocking.